# Project 2: Transformers

This project is part of the NLP module held in the spring of 2026. Three transformer models are compared, answering physical common sense tasks with the PIQA dataset. 

- **Randomly Initialized Transformer** 
- **Pretrained Transformer** not pretrained or finetuned on PIQA dataset
- **LLM (1B+ Parameters)** same hyperparameters

**Dataset**  
"PIQA: Reasoning about Physical Commonsense in Natural Language" — a binary choice task
where a model selects the more physically plausible solution to a given goal.  
Source: [https://arxiv.org/abs/1911.11641](https://arxiv.org/abs/1911.11641)

**Tools** 
- Course Materials
- Documentations (mostly of imported dependencies)
    - apxml
    - NLTK
    - PyTorch
    - skikit-learn
- Claude AI for the following tasks:
    - Helping formulate and clarify reasoning
    - General coding assistance
- Regex101
- Huggingface

**Weights & Biases**  
All experimental runs are logged and published in the
# TODO report

**Notebook structure**
1. Introduction
2. Setup
3. Preprocessing
4. Model
5. Training
6. Evaluation
7. Interpretation


## Setup

In [10]:
!pip install \
    datasets==4.8.4 \
    numpy==2.4.4 \
    datetime==6.0.0 \
    wandb==0.25.1


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [11]:
from datasets import load_dataset
from datetime import datetime
import numpy as np
import wandb
import re

In [12]:
SEED = 42
np.random.seed(SEED)

In [13]:
TS = datetime.now().strftime("%Y%m%d_%H%M%S")

In [14]:
wandb_project = "nlp-project2-piqa"

## Preprocessing

In [15]:
train_split = load_dataset("ybisk/piqa", split="train[:-1000]", revision='refs/convert/parquet')
valid_split = load_dataset("ybisk/piqa", split="train[-1000:]", revision='refs/convert/parquet')
test_split = load_dataset("ybisk/piqa", split="validation", revision='refs/convert/parquet')

### Feature Selection

In [16]:
COL_GOAL = 'goal'
COL_SOL1 = 'sol1'
COL_SOL2 = 'sol2'
COL_LABEL = 'label'

### Filter HTML Elements

In [17]:
# regex source: https://apxml.com/courses/nlp-fundamentals/chapter-1-nlp-text-processing-techniques/handling-text-noise
# verified with: https://regex101.com
regex_pattern = re.compile(r'<[^>]+>', re.IGNORECASE)
html_elements = 0

for split in [train_split, valid_split, test_split]:
    html_elements += len(split.filter(lambda row: regex_pattern.search(row[COL_GOAL])))
    html_elements += len(split.filter(lambda row: regex_pattern.search(row[COL_SOL1])))
    html_elements += len(split.filter(lambda row: regex_pattern.search(row[COL_SOL2])))

print(f"Number of HTML elements found: {html_elements}")

Number of HTML elements found: 0


## Model

### Randomly Initialized Transformer

### Pretrained Transformer

### LLM (1B+ Parameters)

## Training

In [18]:
SKIP_TRAINING = True
MODEL1_NAME = f"random_{TS}"
MODEL2_NAME = f"pretrained_{TS}"
SWEEP_COUNT = 5


In [19]:
training_config = {
    'max_epochs': 30,
    'patience': 5,
}

# use same model architecture in model 1 and 2
# todo config
def create_sweep_config(model_name):
    return {
        "method": "bayes",
        "metric": {"name": f"{model_name}/valid_acc", "goal": "maximize"},
        "parameters": {
            "lr":                  {"values": [1e-3, 1e-4, 1e-5]},
            "weight_decay":        {"values": [1e-3, 1e-4, 1e-5]},
        },
    }

In [20]:
if not SKIP_TRAINING:
    wandb.login()
else: 
    print("training skipped")

training skipped


In [21]:
def train_model(model, config):
    print(f"model: {model} and config: {config}")

In [22]:
def sweep_run_model1():
    with wandb.init(project=wandb_project, config=training_config, group='random') as wandb_run:
        wandb_config = wandb_run.config
        
        run_name = f"{MODEL1_NAME}_lr{wandb.config.lr}_wd{wandb.config.weight_decay}_{wandb_run.id[:4]}"
        
        config = {
            'lr': wandb_config.lr,
            'weight_decay': wandb_config.weight_decay,
            'max_epochs': training_config['max_epochs'],
            'patience': training_config['patience'],
            'model_path': f"models/{run_name}.pt"
        }
        
        model = {}
        
        train_model(model, config)
        
        
if not SKIP_TRAINING:
    sweep_model1 = wandb.sweep(sweep=create_sweep_config(MODEL1_NAME), project=wandb_project)
    
    wandb.agent(sweep_model1, function=sweep_run_model1, count=SWEEP_COUNT)
else: 
    print("training skipped")

training skipped


In [23]:
def sweep_run_model2():
    with wandb.init(project=wandb_project, config=training_config, group='pretrained') as wandb_run:
        wandb_config = wandb_run.config
        
        run_name = f"{MODEL2_NAME}_lr{wandb.config.lr}_wd{wandb.config.weight_decay}_{wandb_run.id[:4]}"
        
        config = {
            'lr': wandb_config.lr,
            'weight_decay': wandb_config.weight_decay,
            'max_epochs': training_config['max_epochs'],
            'patience': training_config['patience'],
            'model_path': f"models/{run_name}.pt"
        }
        
        model = {}
        
        train_model(model, config)
        

if not SKIP_TRAINING:
    sweep_model2 = wandb.sweep(sweep=create_sweep_config(MODEL2_NAME), project=wandb_project)
    
    wandb.agent(sweep_model2, function=sweep_run_model2, count=SWEEP_COUNT)
else: 
    print("training skipped")

training skipped


## Evaluation

## Interpretation